<h1>M45 Gaia Star Cluster Hertzsprung Russel Diagrams (HRD)</h1>

Here are some useful links 
- [European Space Agency Gaia Mission - Writing Queries Turorial](https://www.cosmos.esa.int/web/gaia-users/archive/writing-queries)
- [Gaia's Hertzsprung-Russel Diagram](https://sci.esa.int/web/gaia/-/60198-gaia-hertzsprung-russell-diagram)
- [Measuring the Age of a Star Cluster](https://www.e-education.psu.edu/astro801/content/l7_p6.html#:~:text=The%20HR%20diagram%20for%20stage,%2D13%20billion%20years%20old)
- [Astro Prof YouTube Channel Phys 1403](https://www.youtube.com/watch?v=sDds6c7HByg)

**In the examples below we will query open clusters in the Milky way and plot it in a Hertzsprung-Russel Diagram (HRD). We will also attempt to identify Zero-order Main Sequence (ZAMS) entry points and Main Sequence Cutoff for these star clusters**

*adapted for the Python for Astronomy Course by Chandru Narayan Nov 1, 2022 using the material develped by Dr. Priya Hasan*


In [ ]:
import numpy as np
import pandas as pd
import math
#from sklearn.datasets import make_blobs
#from sklearn.neighbors import NearestNeighbors
#from sklearn.cluster import DBSCAN
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

In [ ]:
# import astroquery and astropy packages
import astropy.units as u
import astropy.coordinates as coord
from astroquery.gaia import Gaia
from astroquery.vizier import Vizier
from astropy import log


In [ ]:
#!echo $PATH  Remove leading '#' for debugging as needed

In [ ]:
# Data Folder
data_folder="data_folder"
!ls -l {data_folder}

In [ ]:
#!pip list  # for debugging as needed

In [ ]:
#EXECUTE THIS CELL THEN RESTART KERNEL
#!pip install astroquery  # for debugging as needed
#!pip install scikit-learn # for debugging or as needed

In [ ]:
# Create Access to the Gaia Database
from astroquery.gaia import Gaia

# Load Tables from the Gai Database
tables = Gaia.load_tables(only_names=True)

# Print the Table Names in the Gaia Database
#for table in tables:
#    print(table.name)

In [ ]:
# Get Gaia Sources Table
meta = Gaia.load_table('gaiadr3.gaia_source')
#print(meta)

In [ ]:
# Print all columns in the Gaia Source Table
for column in meta.columns:
    print(column.name)

## Open Star Cluster M45 in Orion has younger stars
![M45](M45.png)
## Globular Star Cluster M67 in Cancer is VERY OLD!
![M67](M67.png)
## The Gaia Sattelite
Gaia is a space observatory of the European Space Agency, launched in 2013 and expected to operate until 2025. The spacecraft is designed for astrometry: measuring the positions, distances and motions of stars with unprecedented precision.
Gaia  provides astrometry, photometry, and spectroscopy of more than 1000 million stars in the Milky Way. Also data for significant samples of extragalactic and Solar system objects is made available. The Gaia Archive contains deduced positions, parallaxes, proper motions, radial velocities, and brightnesses. Complementary information on multiplicity, photometric variability, and astrophysical parameters is provided for a large fraction of sources.
![gaia](gaia.png)


## Gaia Data

Gaia Data primarily contains of - Right Ascension (RA), Declination (Dec), Parallax, Radial Velocity (RV), Proper Motion in terms of Right Ascension (pmra), and Proper Motion in terms of Declination (pmdec).

1. ✅ **Right Ascension and Declination**- They are the longitude and latitude to position an object in the celestial frame of reference, or they are the celestial coordinates. They are calculated as positions in the plane of the sky. Read more about them at [https://skyandtelescope.org/astronomy-resources/right-ascension-declination-celestial-coordinates/](https://skyandtelescope.org/astronomy-resources/right-ascension-declination-celestial-coordinates/).
[(Image Source)](https://en.wikipedia.org/wiki/Right_ascension)

<img src="https://upload.wikimedia.org/wikipedia/commons/6/66/Ra_and_dec_demo_animation_small.gif" alt="RA Dec" style="width: 50%; float:center;"/> 

2. ✅ **Parallax**- The effect which causes an apparent shift in the position of an object with request to a background when observed from two different points (separated by a distance called basis) It is calculated as the semi-angle of inclination of these two different line of sights from the observation points to the object. Image source and more at: [https://en.wikipedia.org/wiki/Parallax](https://en.wikipedia.org/wiki/Parallax) 

<img src="Parallax_Example.png" alt="Parallax" style="width: 50%; float:center;"/>

3. ✅ **Radial Velocity**- It is the velocity of an object in a direction away from or towards the Earth (observation point). In a more general sense, it is the velocity between the object and the observation point in the direction of the radius connecting the point and the object

4. ✅ **Proper Motions (RA and Dec)**- Proper Motion is the rate of angular drift in the plane of the sky or in a transverse direction. In other words, pmra and pmdec are the rates of change of the RA and Dec of an object in the sky respectively. Their resultant is also called the transverse velocity or total proper motion. The space velocity of an object is the resultant of the transverse velocity and the radial velocity

### Gaia Data Releases

Gaia data is made publicly available through periodic data releases (DRs). Each Data Release has a richer data than the previous data release as Gaia covers the stars more times and adds new stars and objects as well. We had 3 full releases (DR1, DR2, DR3)  Data Release 3 (DR3) is the latest update. We will be working with the most recent full release, DR2. You can also try working with EDR3 with almost negligible changes to the queries we use here.


### Gaia Archive

Gaia Archive is a remote server which hosts the publicly available Dsta Releases of Gaia in the form of a database. It also provides us an interface to query the data and manipulate it according to our needs on the server itself, without us having the need to download the data first on our local computers. Using the Gaia archive site, we can get data on the positions, brightnesses, distances, and more for millions of stars and do various kinds of science and data visualization from them.

**The Gaia archive can be found here: https://gea.esac.esa.int/archive/**

### 

### Review Magnitude and Distance/Parallax Formulas (we drived these in the Star MAgnitudes Module!)
#### The quantity $\boxed{m_{app} - m_{abs}} $ OR $ \boxed {m - M} $ is known as the distance modulus
#### Note that this quantity appears in the equations below to calculate magnitudes and distance
### 1. How to calculate Magnitudes when distance (in pc) is known
#### $$ \boxed{m - M =  5 \times log_{10}(distance) - 5} $$
### 2. How to calculate Magnitudes when parallax (in arc-sec) is known
#### $$ \boxed{m - M =  5 \times log_{10}(1/parallax) - 5} $$
### 3. How to calculate Magnitudes for Gaia calculations when parallax (in milli-arc-sec or 'mas') is known
#### $$ \boxed{m -M = - 5 \times log_{10}(parallax) + 10} $$ OR $$ \boxed{M = m + 5 \times log_{10}(parallax) - 10} $$
### 4. How to calculate Distance (in pc) when apparent and absolute magnitudes are known
#### $$ \boxed{distance = 10^{\frac{m - M+5}{5}}} $$

# STEP A M45: YOU CAN SKIP THIS STEP USE FOR REFERENCE PURPOSES - START AT STEP B BELOW
## Basic Search for Manual Gaia Query and Download
Task: We will use the Basic Search in Gaia Archive to get data of a cluster M45 (Pleiades,  Seven Sisters) in 20 arcminutes radius circle around  it.  We will then read this data in Python and plot the required data.
Steps for Basic Search:

Make sure you're on the Basic query page
In the "Name" field, type in "Messier 45". It should resolve the name. 
To the right, put a "20" and then change the unit from "arc sec" to "arc min". This will tell the archive to search in a radius of 20 arcminutes around M45. There are 60 arcseconds in an arcminute, and 60 arcminutes in a degree. 
Make sure that the "Search In" drop down says "gaiadr3.gaia_source". This specifies the data we want to use is frrom source of Gaia DR3
Click "Submit Query"
You'll see a table pop up with the first 20 results from the query. At the bottom, change "VOTable" to "csv" and click "Download results". This will download a csv to your computer with the queried data in it.

In [ ]:
%%html
<div style="text-align:center;">
<iframe src="https://gea.esac.esa.int/archive/" width="900" height="540"></iframe>
</div>

### Once downloaded, drag and drop downloaded file into the 'data_folder' file panel in Jupyter. 
### Rename file to 'manual_target_name.csv', for example 'manual_m45.csv'.

In [ ]:
# For use with Manual Search

# Drag and drop the downloaded csv file in to the data_folder and rename to m45.csv

# import pandas
import pandas as pd

target = 'm45'
data_file = f'manual_{target}.csv'
file_path = f'{data_folder}/{data_file}'
updated_file_path = f'{data_folder}/updated_{data_file}'

# Now we can read the csv file into a pandas dataframe. 
m45 = pd.read_csv(file_path) # data_file to be found in data_folder

# Checking the top few rows of the data and the number of rows and columns
print("(Rows, Columns) =", m45.shape)
m45.head()

## Let's practice by calculating the Absolute Magnitude of the stars in the M45 csv file we queried earlier and updating the file with it as a new column. We will perform the following steps:
### 1. Read in the 'manual_target_name.csv' file using the np.genfromtxt() method
### 2. Extract relevant columns and print to validate. Notice all the NaN
### 3. Convert array in to a Pandas DataFrame and print it, Validate column headings and values
### 4. Clean out the rows containing NaN using method dropna() and print again. Notice cleaned DF with reduced rows!
### 5. Mask for parallax>0 and update the dataframe and print to validate dataframe
### 6. Calculate the Distance using Parallax & Absolute Magnitude using distance (see cells above for formulae)
### 7. Add these as a new column to existing DF and print to validate
### 9. Write out the results as a new csv file
 

In [ ]:
## DO YOUR WORK STATED ABOVE AS 5-STEPS IN THIS CELL
########################################################################
### STEP 1  YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ????
########################################################################
log.info(f"Getting the DR3 results stored in {file_path}\n")
gc = np.genfromtxt(file_path, delimiter=',', names=True, dtype=None)
########################################################################
### STEP 2
########################################################################
source_id, ra, dec, parallax, pmra, pmdec, bp_rp, phot_g_mean_mag = \
gc['source_id'], gc['ra'], gc['dec'], \
gc['parallax'], gc['pmra'], gc['pmdec'], gc['bp_rp'], gc['phot_g_mean_mag']
########################################################################
### STEP 3
########################################################################
df = pd.DataFrame(gc, columns=['source_id', 'ra', 'dec', 'parallax', 'pmra', 'pmdec', 'bp_rp', 'phot_g_mean_mag'])
print(df)
########################################################################
### STEP 4
########################################################################
df = df.dropna()
print(df)
########################################################################
### STEP 5
########################################################################
mask = (df['parallax']>0)
df= df[mask]
print(df)
########################################################################
### STEP 6    YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################
### When expressed in arc-sec: dist in au = 1/parallax  Type it in below
dist = ????
########################################################################
### See formulas above - pick one to type in below. Abs Mag is 'M' and Apparent Mag is 'm'
### They were all derived from first principles in the python for astronomy class 
### The apparent mag from gaia is 'phot_g_mean_mag' and parallax is 'parallax'
### The log base 10 function is np.log10()
abs_mag = ????
########################################################################
### STEP 7 
########################################################################
df['abs_mag'] = abs_mag
df['dist'] = dist
print(df)
### STEP 9
df.to_csv(updated_file_path, index=False)

# STEP B M45: Now that we have practiced doing this manually we will automate this process
## by setting up an API cone search using Gaia
## We will then use the Proper Motion  in RA and DEC to filter (mask) foreground stars
### We will then Plot all of the stars in a Hertzsprung Russell Diagram Abs Mag vs Temp (Color)

## Setup for API Gaia Target Cone Search and Download

In [ ]:
# Parameters for API Query & HRD (20 minutes radius around M45 center) - Change for each query!
target = "M45" # target to query
object_radius = 1/5 * u.deg # 20 arc-min
coordinate = coord.SkyCoord.from_name(target)
print(coordinate)
num_stars = 100000 #maximum number of stars to retrieve

# Setup file paths
automated_data_file = f'automated_{target}_{num_stars}.csv'
automated_file_path = f'{data_folder}/{automated_data_file}'
print(automated_file_path)

In [ ]:
# Query Gaia
query_sub = f"""
    source_id, ra, dec, parallax, pmra, pmdec, bp_rp, phot_g_mean_mag,  
    phot_g_mean_mag+5*log10(parallax)-10 as amg, 1000/parallax as dist
    FROM gaiaedr3.gaia_source 
    WHERE parallax>0 AND bp_rp > -0.75 AND bp_rp < 2 AND 1=CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {coordinate.ra.value}, {coordinate.dec.value}, {object_radius.value})
    )
""" 

query_full = f"""
    source_id, ra, dec, parallax, pmra, pmdec, bp_rp, phot_g_mean_mag
    FROM gaiaedr3.gaia_source 
    WHERE 1=CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {coordinate.ra.value}, {coordinate.dec.value}, {object_radius.value})
    )
""" 

#query = f" SELECT TOP {query_size} {query_full}"
query = f" SELECT {query_full}"

print(query)

try:
    log.info(f"Getting the DR3 results stored in {automated_file_path}\n")
    
    #gc = np.recfromcsv(automated_file_path)
    gc = np.genfromtxt(automated_file_path, delimiter=',', names=True, dtype=None)
    source_id, ra, dec, parallax, pmra, pmdec, bp_rp, phot_g_mean_mag = \
    gc['source_id'], gc['ra'], gc['dec'], \
    gc['parallax'], gc['pmra'], gc['pmdec'], gc['bp_rp'], gc['phot_g_mean_mag']
    
    print(f"reading results from previously existng {automated_data_file}\n")
    #print(ra, dec, plx, amg, dist)
    df = pd.DataFrame({"source_id": source_id, "ra": ra, "dec": dec, "parallax": parallax, "pmra": pmra, "pmdec": pmdec, "bp_rp": bp_rp, "phot_g_mean_mag": phot_g_mean_mag})
    

except OSError:
    log.info(f"Performing new query from Gaia as previous query {automated_data_file} does not exist\n")

    Gaia.ROW_LIMIT = -1
    job = Gaia.launch_job_async(query, dump_to_file=True, output_format="csv",
                                 output_file=automated_file_path)
    print(job)
    r = job.get_results()
    df = r.to_pandas()


In [ ]:
#################################################################################
### STEP     USES FORMULAS STATED ABOVE. DERIVED IN python for astronomy SEMINAR
#################################################################################
# Add the calculated columns abs_mag and dist to dataframe.
########################################################################
########################################################################
### STEP 6    YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################
df['abs_mag'] = ????
df['dist'] = ????
df

In [ ]:
## save the ASCII table as a pandas dataframe
all_stars = df
type(all_stars)
print(all_stars)

In [ ]:
## plotting the skyplot 

#fig, axs = plt.subplots(1)
#plt.margins(0.005, tight=True)
#axs.set_aspect('equal')
sns.set(rc={'figure.figsize':(9,9)})
skyplot = sns.scatterplot(x='ra', y='dec', data = all_stars)
#skyplot = sns.scatterplot(x='ra', y='dec', data = members)
skyplot.invert_xaxis()
plt.title('Skyplot of ' + target + ' data')
plt.show()

########################################################################
### QUESTION     WHY IS THIS CIRCULAR ????
########################################################################

In [ ]:
########################################################################
### STEP     PLOT YOUR OPEN CLUSTER'S HRD - YOUR FIRST !!!
### QUESTION I SEE THE MAIN SEQUENCE - WHAT IS ALL THAT OTHER CRAP ? pardon my french ...
########################################################################
sns.scatterplot(x = 'bp_rp', y='phot_g_mean_mag',palette='RdYlGn',
                data = all_stars)
plt.ylabel('Apparent Magnitude (G band)')
#plt.xlim(0,3)
#sns.scatterplot(hr.b_v+0.5, hr.V+10)

plt.gca().invert_yaxis()

In [ ]:
########################################################################
### STEP     LET'S ELIMINATE ANY STARS NOT BELONGING TO M45 !!!
###          BY ANALYZING WITH MORE PLOTS WITH PROPER MOTIONS
### QUESTION WHAT DO YOU SEE ?
########################################################################
import seaborn as sns
sns.set(rc={'figure.figsize':(8.7,6.27)})

#sns.scatterplot(x='pmra', y='pmdec', data=features[features.labels > -1], hue='labels',legend='full')

sns.scatterplot(x='pmra', y='pmdec', data=all_stars)

In [ ]:
########################################################################
### STEP     LET'S ZOOM IN WHERE THERE ARE STARS
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################         
### QUESTION WHAT DO YOU SEE ?
########################################################################
# limit ranges to zoom in
plt.xlim(-50,50)
plt.ylim(????,????)
sns.scatterplot(x='pmra', y='pmdec', data=all_stars)

In [ ]:
########################################################################
### STEP     LET'S LOOK AT IT A DIFFERENT WAY
###          BY PLOTTING PARALLAX AGAINST PMRA
###          ALSO ZOOM IN!!
########################################################################
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################            
### QUESTION WHAT DO YOU SEE ?
########################################################################
sns.scatterplot(x='pmra', y='parallax', data=all_stars)
plt.xlim([????, ????])

In [ ]:
########################################################################
### STEP     LET'S LOOK AT IT A DIFFERENT WAY
###          BY PLOTTING PARALLAX AGAINST PMDEC
###          ALSO ZOOM IN!!
########################################################################
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################    
sns.scatterplot(x='pmdec', y='parallax', data=all_stars)
plt.xlim([????,????])

In [ ]:
########################################################################
### STEP     LET'S LOOK AT IT A DIFFERENT WAY
###          BY PLOTTING HISTOGRAM OF PARALLAX
########################################################################
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################   
sns.histplot(x='parallax', data=all_stars,
              kde=True,color='blue')

plt.xlim([????,????])
plt.legend()
plt.show()

In [ ]:
########################################################################
### STEP     LET'S LOOK AT IT A DIFFERENT WAY
###          BY PLOTTING HISTOGRAM OF PMRA
########################################################################
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################  
sns.histplot(x='pmra', data=all_stars,
              kde=True,color='blue')
plt.xlim([????,????])
plt.legend()
plt.show()

In [ ]:
########################################################################
### STEP     LET'S LOOK AT IT A DIFFERENT WAY
###          BY PLOTTING HISTOGRAM OF PMDEC
########################################################################
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################  
########################################################################
### STEP     LET'S LOOK AT IT A DIFFERENT WAY
###          BY PLOTTING HISTOGRAM OF PMDEC
########################################################################
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################  
sns.histplot(x='pmdec', data=all_stars,
              kde=True,color='blue')
plt.xlim([????,????])
plt.legend()
plt.show()

In [ ]:
all_stars.describe()

In [ ]:
all_stars.info()

In [ ]:
########################################################################
### STEP     CREATE A MASK BY SETTING LIMITS FOR all_stars DATAFRAME
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################  
mask = (all_stars.parallax >= 5) & (all_stars.parallax <= ????) & \
    (all_stars.pmra >= ????) & (all_stars.pmra <= ????) & \
    (all_stars.pmdec >= ????) & (all_stars.pmdec <= ????)  


In [ ]:
########################################################################
### STEP     CALCULATE THE ABS MAG FOR all_stars IN DATAFRAME
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################  
abs_mag = all_stars['phot_g_mean_mag']+5*np.log10(all_stars['parallax'])-10
print(abs_mag)

In [ ]:
########################################################################
### STEP     ADD THE ABS MAG TO THE all_stars DATAFRAME
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
########################################################################   
all_stars['abs_mag']=abs_mag

In [ ]:
########################################################################
### STEP     APPLY MASK YOU CREATED all_stars DATAFRAME SAVE IN members
######################################################################## 
members = all_stars[mask]

In [ ]:
members.describe()
members.info()

In [ ]:
print(members)

In [ ]:
members_clean = members.dropna()
print(members_clean)

In [ ]:
########################################################################
### STEP     PLOT HRD FOR members AND all_stars in 2 colors
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
######################################################################## 
sns.scatterplot(x = 'bp_rp', y='????',palette='RdYlGn',
                data = all_stars)
sns.scatterplot(x = 'bp_rp', y='????',palette='RdYlGn',
                data = members_clean)
plt.ylabel('Apparent Magnitude (G band)')
#plt.xlim(0,3)
#sns.scatterplot(hr.b_v+0.5, hr.V+10)

plt.gca().invert_yaxis()

In [ ]:
########################################################################
### STEP     PLOT HRD FOR all_stars & members USING ABSOLUTE MAGNITUDE VS BP_RP
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
######################################################################## 
sns.scatterplot(x = 'bp_rp', y='????',palette='RdYlGn',
                data = all_stars)
sns.scatterplot(x = 'bp_rp', y='????',palette='RdYlGn',
                data = members_clean)
plt.ylabel('Absolute Magnitude (G band)')
#plt.xlim(0,3)
plt.ylim(-2.5,10)
#sns.scatterplot(hr.b_v+0.5, hr.V+10)

plt.gca().invert_yaxis()

In [ ]:
########################################################################
### STEP     PLOT HRD FOR members only USING ABSOLUTE MAGNITUDE VS BP_RP
### STEP     YOUR INPUTS ARE NEEDED HERE WHERE YOU SEE ???? !!!
######################################################################## 
sns.scatterplot(????)
plt.ylabel('Absolute Magnitude (G band)')
#plt.xlim(0,3)
#sns.scatterplot(hr.b_v+0.5, hr.V+10)
plt.gca().invert_yaxis()
plt.title('Hertzsprung Russel Diagram of ' + target)
out_file="{}/{}_HRD.png".format(data_folder,target)
plt.savefig(out_file)

## Can you tell where the Main Sequence Turnoff is happening?
## What does that tell you about the Age of the cluster?
## [This excellent article describes it all - PLEASE READ!](https://courses.ems.psu.edu/astro801/content/l7_p6.html)
![age_turnoff_point.png](age_turnoff_point.png)
## Answer Questions below

## What is the turnoff Point (Oldest Star still on the Main Sequence) for Open Cluster M45 ??  
### IT IS AROUND ABS MAG of ????
### From this M45's age can be predicted to be around ???? Years!

#### Your Assignment: Plot HRD for a few other clusters M67, M44, NGC1893, NGC581, M92, M13) to see the similarities and differences in HRDs.
